# 02 Data Cleaning

This notebook reshapes the FAOSTAT-style long dataset into a clean analytical table.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'crop_production_raw_faostat_style_synthetic.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'crop_production_cleaned_from_notebook.csv'

raw_df = pd.read_csv(RAW_PATH)
raw_df.columns = raw_df.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)
raw_df.head()

In [ ]:
metric_map = {
    'Area harvested': 'area_harvested_ha',
    'Yield': 'yield_kg_per_ha',
    'Production': 'production_tonnes',
}

filtered_df = raw_df[raw_df['element'].isin(metric_map)].copy()
filtered_df['metric'] = filtered_df['element'].map(metric_map)

cleaned_df = (
    filtered_df
    .pivot_table(
        index=['country', 'iso3_code', 'year', 'crop'],
        columns='metric',
        values='value',
        aggfunc='sum'
    )
    .reset_index()
)
cleaned_df.columns.name = None
cleaned_df.head()

In [ ]:
cleaned_df.isna().sum()

In [ ]:
cleaned_df.duplicated(subset=['country', 'year', 'crop']).sum()

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH